In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raesiow/nova-tf-idf/cm_tfidf.png
/kaggle/input/datasets/raesiow/nova-tf-idf/baseline_preds.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/cm_text_vs_both.png


In [2]:
import torch                                  # PyTorch, the library DistilBERT runs on

print(torch.cuda.is_available())              # expect True (GPU is on); if False, check Settings -> Accelerator
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")   # e.g. 'Tesla T4'


True
Tesla T4


In [3]:
path = '/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv'
df = pd.read_csv(path, dtype={"off_code": str, "fpx_code": str})   # keep barcodes as text

print(df.shape)                                        # expect (1884, 22)
print(pd.crosstab(df["fold"], df["nova"]))             # expect the same table as last night (fold 0: 108 / 13 / 53 / 202, ...)

(1884, 22)
nova    1   2   3    4
fold                  
0     108  13  53  202
1      95  11  58  214
2     101  12  46  231
3      82  13  56  209
4      91  12  52  225


In [4]:
from transformers import AutoTokenizer   # Hugging Face tool that loads the matching tokenizer for a model

model_name = "distilbert-base-uncased"                   # "uncased" = lowercases text first, like our TF-IDF did
tok = AutoTokenizer.from_pretrained(model_name)          # download the tokenizer (needs Internet ON)

print(tok.tokenize("sugar, sucralose, emulsifier (soy lecithin)"))   # see how one example gets chopped up

lengths = df["ingredients_text"].apply(lambda t: len(tok(t)["input_ids"]))   # count tokens in every ingredient text
print(lengths.describe())                                # min, average, max token counts
print((lengths > 256).mean())                            # share of texts longer than 256 tokens

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

['sugar', ',', 'su', '##cr', '##alo', '##se', ',', 'em', '##uls', '##ifier', '(', 'soy', 'le', '##cit', '##hin', ')']


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (893 > 512). Running this sequence through the model will result in indexing errors


count    1884.000000
mean       64.508493
std        68.191613
min         3.000000
25%        13.750000
50%        45.000000
75%        96.000000
max       893.000000
Name: ingredients_text, dtype: float64
0.01910828025477707


In [5]:
from torch.utils.data import Dataset    # the base class PyTorch expects

df["label"] = df["nova"] - 1             # PyTorch wants classes numbered 0,1,2,3 (not 1,2,3,4); NOVA 1 -> 0, ..., NOVA 4 -> 3
MAX_LEN = 256                            # our cut-off from the length check

class NovaDataset(Dataset):                                   # our recipe, built on PyTorch's Dataset
    def __init__(self, texts, labels):                        # runs once when we create it
        self.enc = tok(list(texts),                           # tokenise every text at once
                       truncation=True,                       # cut texts longer than MAX_LEN
                       max_length=MAX_LEN,
                      return_token_type_ids = False)
        self.labels = list(labels)                            # store the labels as a plain list

    def __len__(self):                                        # "how many items?"
        return len(self.labels)

    def __getitem__(self, i):                                 # "give me item i"
        item = {key: torch.tensor(val[i])                     # token IDs + attention mask for row i, as tensors
                for key, val in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])         # the answer for row i
        return item

ds = NovaDataset(df["ingredients_text"][:3], df["label"][:3])   # small test: just the first 3 rows
print(len(ds))                                                  # expect 3
print(ds[0])                                                    # expect a dict: input_ids, attention_mask, labels
print(tok.decode(ds[0]["input_ids"]))                           # turn the IDs back into text, to check it worked

3
{'input_ids': tensor([  101,  2878,  8982,  1051, 11149,  1010,  5699,  1010,  9781,  2732,
         2818,  1010,  6861,  1010,  2829,  5699, 23353,  1010,  5474,  1010,
         4440, 17287, 18719,  2819, 17344,  1010,  2064,  6030,  1998,  1013,
         2030,  3103, 14156,  3514,  1010,  3019, 26011, 14894,  1010, 17663,
         1041,  1006,  3816,  2000,  3597, 27921, 27896,  1007,  1010, 13853,
        26427,  1010, 17663,  1039,  1006, 13365,  2004, 27108, 20179,  1007,
         1010,  3707,  1998, 15813,  1006,  9754, 20435,  1007,  1010,  1037,
         1038, 17663,  1006,  9152,  6305,  3981, 24284,  1007,  1010, 17663,
         1038,  2575,  1006,  1052, 12541, 13820, 20303,  2063, 18479,  2818,
        10626,  5178,  1007,  1010, 17663,  1038, 11871,  1006, 16215, 25107,
         2378, 18847,  3490,  6494,  2618,  1007,  1010, 17663,  1037,  1006,
         5340, 17570,  1007,  1010, 17663,  1038,  2475,  1006, 19395, 11253,
        14973,  2378,  1007,  1010,  1037,  1038

In [6]:
from torch.utils.data import DataLoader                                  # serves the Dataset in batches
from transformers import DataCollatorWithPadding, AutoModelForSequenceClassification

device = torch.device("cuda")                       # the GPU
collate = DataCollatorWithPadding(tok)              # pads each batch to its own longest text (with 0s in attention_mask)

small = NovaDataset(df["ingredients_text"][:8], df["label"][:8])          # 8 rows for the test
loader = DataLoader(small, batch_size=4, collate_fn=collate)              # serve them 4 at a time, padded

batch = next(iter(loader))                          # grab the first batch only
print({k: v.shape for k, v in batch.items()})       # expect input_ids & attention_mask (4, some length), labels (4,)

model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=4).to(device)    # pretrained DistilBERT + a new 4-class output layer, moved to GPU

batch = {k: v.to(device) for k, v in batch.items()} # move the batch to the GPU too (model and data must be in the same place)
with torch.no_grad():                               # just testing, so don't track gradients (faster, less memory)
    out = model(**batch)                            # run the model; ** unpacks the dict into named inputs

print(out.logits.shape)                             # expect torch.Size([4, 4]): 4 texts x 4 class scores
print(out.loss)                                     # expect around 1.39 (= log 4: an untrained model is guessing evenly)


{'input_ids': torch.Size([4, 142]), 'attention_mask': torch.Size([4, 142]), 'labels': torch.Size([4])}


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([4, 4])
tensor(1.4135, device='cuda:0')


In [7]:
from torch.optim import AdamW                                    # the optimiser: decides how big each weight nudge is
from transformers import get_linear_schedule_with_warmup         # learning-rate schedule: ramp up, then slowly down
from sklearn.utils.class_weight import compute_class_weight      # computes "balanced" class weights
from sklearn.metrics import f1_score, classification_report      # scoring
import time                                                      # to time each fold

del model                         # free the test model from Cell 5
torch.cuda.empty_cache()          # give its GPU memory back

EPOCHS = 10 #4                        # passes over the training data
BATCH = 16                        # texts per training step
LR = 2e-5                         # learning rate: size of each nudge (small, so we don't wreck the pretrained knowledge)

def train_fold(k, seed=42):
    torch.manual_seed(seed)                                      # fix randomness (new layer's start, shuffling) so reruns match

    train = df[df["fold"] != k]                                  # 4 training folds
    test  = df[df["fold"] == k]                                  # 1 test fold

    train_loader = DataLoader(NovaDataset(train["ingredients_text"], train["label"]),
                              batch_size=BATCH, shuffle=True,    # shuffle = new random order each epoch
                              collate_fn=collate)
    test_loader  = DataLoader(NovaDataset(test["ingredients_text"], test["label"]),
                              batch_size=64, shuffle=False,      # bigger batches are fine when not training; keep order
                              collate_fn=collate)

    model = AutoModelForSequenceClassification.from_pretrained(
                model_name, num_labels=4).to(device)             # FRESH pretrained model every fold (no memory of other folds)

    w = compute_class_weight("balanced", classes=np.arange(4),   # rarer class -> bigger weight (same idea as the baseline)
                             y=train["label"])
    loss_fn = torch.nn.CrossEntropyLoss(
                weight=torch.tensor(w, dtype=torch.float).to(device))   # mistakes on NOVA 2 now cost more

    opt = AdamW(model.parameters(), lr=LR)                       # optimiser over all of DistilBERT's weights
    steps = EPOCHS * len(train_loader)                           # total training steps
    sched = get_linear_schedule_with_warmup(opt,
                int(0.1 * steps), steps)                         # first 10% of steps: LR rises from 0; then falls to 0

    for epoch in range(EPOCHS):                                  # ---- training ----
        model.train()                                            # training mode (turns on dropout)
        total = 0                                                # running sum of loss, for printing
        for batch in train_loader:
            batch = {key: v.to(device) for key, v in batch.items()}   # move batch to GPU
            labels = batch.pop("labels")                         # take the answers out (we compute weighted loss ourselves)
            logits = model(**batch).logits                       # 1. forward: class scores
            loss = loss_fn(logits, labels)                       # 2. loss: how wrong, with class weights
            loss.backward()                                      # 3. backward: compute gradients
            opt.step()                                           # 4. step: nudge the weights
            sched.step()                                         #    move the learning-rate schedule along
            opt.zero_grad()                                      # 5. clear gradients
            total += loss.item()                                 # add this batch's loss (as a plain number)
        print(f"fold {k} epoch {epoch+1}: train loss {total/len(train_loader):.3f}")   # average loss this epoch

    model.eval()                                                 # ---- testing (once, at the end) ----
    probs = []
    with torch.no_grad():                                        # no gradients needed
        for batch in test_loader:
            batch = {key: v.to(device) for key, v in batch.items()}
            batch.pop("labels")                                  # model doesn't need the answers to predict
            logits = model(**batch).logits
            probs.append(torch.softmax(logits, dim=1).cpu())     # softmax turns scores into probabilities that add to 1
    probs = torch.cat(probs).numpy()                             # stick batches together -> (test rows, 4) array

    del model                                                    # free GPU memory before the next fold
    torch.cuda.empty_cache()
    return test.index, probs                                     # which rows, and their 4 NOVA probabilities

In [8]:
t0 = time.time()                                       # start the timer
idx, probs = train_fold(0)                             # train on folds 1-4, predict fold 0
pred = probs.argmax(axis=1)                            # pick the class with the highest probability in each row
print(f1_score(df.loc[idx, "label"], pred, average="macro"))   # macro-F1 on fold 0; the TF-IDF text-only score here was 0.667
print(f"{time.time() - t0:.0f} seconds")              # how long one fold takes

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 0 epoch 1: train loss 1.271
fold 0 epoch 2: train loss 0.935
fold 0 epoch 3: train loss 0.624
fold 0 epoch 4: train loss 0.458
fold 0 epoch 5: train loss 0.323
fold 0 epoch 6: train loss 0.270
fold 0 epoch 7: train loss 0.185
fold 0 epoch 8: train loss 0.150
fold 0 epoch 9: train loss 0.127
fold 0 epoch 10: train loss 0.105
0.766543950695223
263 seconds


In [9]:
oof = np.zeros((len(df), 4))                     # "out-of-fold" probabilities: one row per product, 4 columns (NOVA 1-4)
scores = []                                      # macro-F1 per fold
t0 = time.time()

for k in range(5):                               # each fold takes a turn as test
    idx, probs = train_fold(k)                   # fresh model, trained on the other 4 folds
    oof[df.index.get_indexer(idx)] = probs       # put this fold's probabilities into the right rows
    pred = probs.argmax(axis=1)                  # predicted class (0-3)
    f1 = f1_score(df.loc[idx, "label"], pred, average="macro")
    scores.append(f1)
    print(f"fold {k}: macro-F1 = {f1:.3f}   ({time.time() - t0:.0f} s so far)")

print(f"DistilBERT text-only: mean = {np.mean(scores):.3f} ± {np.std(scores):.3f}")
# compare: TF-IDF text-only 0.638 ± 0.033, TF-IDF text + nutrients 0.700 ± 0.037
# fold 0 should be 0.752 again (same seed); tiny differences are possible on GPU

out = df[["fpx_code", "off_code", "nova", "fold"]].copy()          # IDs, true label, fold
out[["p1", "p2", "p3", "p4"]] = oof                                # DistilBERT's probability for each NOVA class
out.to_csv("/kaggle/working/distilbert_oof.csv", index=False)      # save; we need these for "+ nutrients" and the fused MLP
print(out.shape)                                                   # expect (1884, 8)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 0 epoch 1: train loss 1.271
fold 0 epoch 2: train loss 0.935
fold 0 epoch 3: train loss 0.624
fold 0 epoch 4: train loss 0.458
fold 0 epoch 5: train loss 0.323
fold 0 epoch 6: train loss 0.270
fold 0 epoch 7: train loss 0.185
fold 0 epoch 8: train loss 0.150
fold 0 epoch 9: train loss 0.127
fold 0 epoch 10: train loss 0.105
fold 0: macro-F1 = 0.767   (274 s so far)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 1 epoch 1: train loss 1.271
fold 1 epoch 2: train loss 0.968
fold 1 epoch 3: train loss 0.710
fold 1 epoch 4: train loss 0.527
fold 1 epoch 5: train loss 0.370
fold 1 epoch 6: train loss 0.269
fold 1 epoch 7: train loss 0.218
fold 1 epoch 8: train loss 0.170
fold 1 epoch 9: train loss 0.151
fold 1 epoch 10: train loss 0.134
fold 1: macro-F1 = 0.704   (550 s so far)


... (5 outputs left out)

fold 3 epoch 1: train loss 1.271
fold 3 epoch 2: train loss 0.964
fold 3 epoch 3: train loss 0.649
fold 3 epoch 4: train loss 0.442
fold 3 epoch 5: train loss 0.310
fold 3 epoch 6: train loss 0.242
fold 3 epoch 7: train loss 0.176
fold 3 epoch 8: train loss 0.137
fold 3 epoch 9: train loss 0.114
fold 3 epoch 10: train loss 0.105
fold 3: macro-F1 = 0.673   (1093 s so far)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 4 epoch 1: train loss 1.260
fold 4 epoch 2: train loss 0.935
fold 4 epoch 3: train loss 0.669
fold 4 epoch 4: train loss 0.440
fold 4 epoch 5: train loss 0.283
fold 4 epoch 6: train loss 0.224
fold 4 epoch 7: train loss 0.166
fold 4 epoch 8: train loss 0.124
fold 4 epoch 9: train loss 0.111
fold 4 epoch 10: train loss 0.101
fold 4: macro-F1 = 0.612   (1366 s so far)
DistilBERT text-only: mean = 0.700 ± 0.055
(1884, 8)


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

nut_cols = ["protein_g_f", "fat_g_f", "carb_excl_fiber_g_f", "sugars_g_f",
            "fiber_g_f", "sat_fat_g_f", "sodium_g_f"]                 # same 7 nutrients as the baseline

feat = np.hstack([np.log(oof + 1e-6),                                 # DistilBERT's 4 probabilities, logged (spreads out values near 0 and 1); 1e-6 avoids log(0)
                  np.log1p(df[nut_cols].values)])                     # + 7 nutrients, log(1 + grams) as before -> 11 columns

scores_stack = []
for k in range(5):                                                    # same 5 folds again
    tr = (df["fold"] != k).values                                     # True for training rows
    te = (df["fold"] == k).values                                     # True for test rows
    scaler = StandardScaler().fit(feat[tr])                           # learn mean/spread from training rows only
    clf = LogisticRegression(class_weight="balanced", max_iter=2000)
    clf.fit(scaler.transform(feat[tr]), df["label"][tr])              # train
    pred = clf.predict(scaler.transform(feat[te]))                    # predict test fold
    f1 = f1_score(df["label"][te], pred, average="macro")
    scores_stack.append(f1)
    print(f"fold {k}: macro-F1 = {f1:.3f}")

print(f"DistilBERT + nutrients: mean = {np.mean(scores_stack):.3f} ± {np.std(scores_stack):.3f}")
# compare with TF-IDF + nutrients: 0.700 ± 0.037

fold 0: macro-F1 = 0.783
fold 1: macro-F1 = 0.741
fold 2: macro-F1 = 0.655
fold 3: macro-F1 = 0.743
fold 4: macro-F1 = 0.667
DistilBERT + nutrients: mean = 0.718 ± 0.049


In [11]:
rng = np.random.default_rng(0)                         # fixed random generator so the shuffle is repeatable
blocks = {"text (DistilBERT probs)": slice(0, 4),      # columns 0-3 of feat = DistilBERT's 4 probabilities
          "nutrients":               slice(4, 11)}     # columns 4-10 = the 7 nutrients

for name, cols in blocks.items():                      # test each block separately
    drops = []
    for k in range(5):                                 # same 5 folds
        tr = (df["fold"] != k).values                  # training rows
        te = ~tr                                       # test rows
        sc = StandardScaler().fit(feat[tr])            # scaling from training rows only
        clf = LogisticRegression(class_weight="balanced", max_iter=2000).fit(sc.transform(feat[tr]), df["label"][tr])
        X_te = sc.transform(feat[te])
        base = f1_score(df["label"][te], clf.predict(X_te), average="macro")   # normal score

        X_shuf = X_te.copy()                           # copy, so the original isn't changed
        X_shuf[:, cols] = X_shuf[rng.permutation(len(X_shuf))][:, cols]       # give each product another product's values for this block
        shuf = f1_score(df["label"][te], clf.predict(X_shuf), average="macro") # score with that block scrambled
        drops.append(base - shuf)                      # how much the score fell

    print(f"{name:25s} drop in macro-F1 when scrambled: {np.mean(drops):.3f}")

text (DistilBERT probs)   drop in macro-F1 when scrambled: 0.332
nutrients                 drop in macro-F1 when scrambled: 0.200


In [12]:
oof = pd.read_csv("/kaggle/working/distilbert_oof.csv")[["p1", "p2", "p3", "p4"]].values   # saved DistilBERT probabilities
df["label"] = df["nova"] - 1                                                             # labels 0-3
nut_cols = ["protein_g_f", "fat_g_f", "carb_excl_fiber_g_f", "sugars_g_f",
            "fiber_g_f", "sat_fat_g_f", "sodium_g_f"]                                    # the 7 nutrients
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer          # TF-IDF, as in the baseline
from sklearn.ensemble import HistGradientBoostingClassifier           # gradient boosting: many small trees

# TF-IDF out-of-fold probabilities (same 5 folds; each row predicted by a model that never saw it)
tfidf_oof = np.zeros((len(df), 4))
for k in range(5):
    tr = (df["fold"] != k).values                                     # training rows
    te = ~tr                                                          # test rows
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    X_tr = vec.fit_transform(df.loc[tr, "ingredients_text"])          # vocabulary from training rows only
    X_te = vec.transform(df.loc[te, "ingredients_text"])
    lr = LogisticRegression(class_weight="balanced", max_iter=2000).fit(X_tr, df.loc[tr, "label"])
    tfidf_oof[te] = lr.predict_proba(X_te)

nut = np.log1p(df[nut_cols].values)                                   # 7 nutrients, log(1 + grams)
feature_sets = {
    "DistilBERT + nutrients":          np.hstack([np.log(oof + 1e-6), nut]),                               # 11 columns
    "TF-IDF + DistilBERT + nutrients": np.hstack([np.log(tfidf_oof + 1e-6), np.log(oof + 1e-6), nut]),     # 15 columns
}
models = {                                                            # settings fixed in advance: run once, no adjusting
    "linear (LR)": lambda: LogisticRegression(class_weight="balanced", max_iter=2000),
    "trees (GBM)": lambda: HistGradientBoostingClassifier(class_weight="balanced", max_iter=200,
                                                          learning_rate=0.1, random_state=100),
}

for fname, F in feature_sets.items():                                 # each set of inputs
    for mname, make in models.items():                                # each final model
        s = []
        for k in range(5):
            tr = (df["fold"] != k).values
            te = ~tr
            sc = StandardScaler().fit(F[tr])                          # scaling from training rows (trees don't need it, but harmless)
            m = make().fit(sc.transform(F[tr]), df["label"][tr])
            s.append(f1_score(df["label"][te], m.predict(sc.transform(F[te])), average="macro"))
        print(f"{fname:32s} {mname}: {np.mean(s):.3f} ± {np.std(s):.3f}   folds {np.round(s, 3)}")
# reference: TF-IDF + nutrients (LR, all in one) = 0.700 ± 0.037

DistilBERT + nutrients           linear (LR): 0.718 ± 0.049   folds [0.783 0.741 0.655 0.743 0.667]
DistilBERT + nutrients           trees (GBM): 0.761 ± 0.038   folds [0.801 0.805 0.761 0.725 0.712]
TF-IDF + DistilBERT + nutrients  linear (LR): 0.719 ± 0.040   folds [0.761 0.717 0.691 0.766 0.662]
TF-IDF + DistilBERT + nutrients  trees (GBM): 0.762 ± 0.056   folds [0.824 0.812 0.777 0.719 0.678]


# 